### This is an example to demonstrate prompt engineering with structured output

You are a support-ticket classifier.

Classify the ticket into exactly one category:
- billing
- technical
- account
- other

Return only valid JSON matching this schema:

{
  "category": "billing | technical | account | other",
  "priority": "low | medium | high",
  "summary": "string, maximum 15 words"
}

Ticket:
“I was charged twice for my Pro subscription and need a refund.”

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI 

load_dotenv()

client = OpenAI()


def llm_call(input_data):
    llm_response = client.responses.create(
                model="gpt-4o-mini",
                tools=tools,
                tool_choice={"type": "function", "name": "classify_ticket"},
                input=input_data              
    )
    return llm_response

def llm_response_parse(llm_response):
    for item in llm_response.output:
        if item.type == "function_call":
            print(item.arguments)

tools =[
    {
        "type": "function",
        "name": "classify_ticket",
        "description": "Classify the tickets in to category",
        # The model must follow your function’s JSON schema exactly.
        "strict": True,
        "parameters": {
            "type":"object",
            "properties": {
                "category": {
                    "type":"string",
                    "enum":["billing","technical","account","other"]
                },
                "priority":{
                    "type": "string",
                    "enum":["low","medium","high"]
                },
                "summary":{
                    "type":"string",
                    "maxLength": 150
                }
            },
            "required": ["category", "priority", "summary"],
            "additionalProperties": False
        },
    }
]

input_data = [{"role":"user","content":"I was charged twice for my Pro subscription and need a refund."}]

llm_response = llm_call(input_data)
llm_response_parse(llm_response)

input_data = [{"role":"developer","content":"I have fixed the sunscription issue."}]

llm_response = llm_call(input_data)
llm_response_parse(llm_response)

{"category":"billing","priority":"high","summary":"Charged twice for Pro subscription and need a refund."}
{"category":"account","priority":"medium","summary":"Fixed the subscription issue."}
